In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("GOOGLE_API_KEY"))  # should print your key

Google API:

In [ ]:
from dotenv import load_dotenv
from google import genai
import os
import base64
import time
from pathlib import Path

# Load API key
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Folders
input_folder = Path("raw_images")
output_folder = Path("ocr_output_Gemini")
output_folder.mkdir(exist_ok=True)

supported = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}
images = sorted([f for f in input_folder.iterdir() if f.suffix.lower() in supported])
print(f"Found {len(images)} images to process.")

for i, image_path in enumerate(images, 1):
    output_path = output_folder / (image_path.stem + ".txt")

    if output_path.exists():
        print(f"[{i}/{len(images)}] Skipping {image_path.name} (already done)")
        continue

    print(f"[{i}/{len(images)}] Processing {image_path.name}...")

    try:
        image_data = base64.b64encode(image_path.read_bytes()).decode("utf-8")
        mime_type = "image/jpeg" if image_path.suffix.lower() in {".jpg", ".jpeg"} else "image/png"

        response = client.models.generate_content(
            model="gemma-4-26b-a4b-it",
            contents=[
                {
                    "parts": [
                        {"inline_data": {"mime_type": mime_type, "data": image_data}},
                        {"text": "Transcribe all text from this image exactly as it appears. \n"
                        "Preserve the layout, column structure, and all numbers precisely."}
                    ]
                }
            ]
        )

        output_path.write_text(response.text, encoding="utf-8")
        print(f"    → Saved to {output_path.name}")

    except Exception as e:
        print(f"    ✗ Error on {image_path.name}: {e}")

    time.sleep(15)  # wait 15 seconds between requests to stay under 10 RPM

print("Done!")